# 🧠 SHIV-AI ULTIMATE v5.0
## Search + Vision + Voice + Coding Expert + Training + RAG

```
🎤 Voice Input     👁️ Image Input     ⌨️ Text Input
        \               |               /
         \              |              /
          ▼             ▼             ▼
    ┌─────────────────────────────────────┐
    │         🧠 SHIV-AI MASTER BRAIN     │
    │                                     │
    │  🔍 Search  +  🗄️ RAG  +  🏋️ Train │
    └─────────────────────────────────────┘
          /             |              \
         /              |               \
        ▼               ▼               ▼
   🔊 Voice Out    📝 Text Out    💻 Code Out
```

| Feature | Status |
|---------|--------|
| 🔍 Online Search (Wikipedia/News/Web) | ✅ |
| 👁️ Vision — Image/Screenshot समझना | ✅ |
| 🎤 Voice Input (बोलकर पूछो) | ✅ |
| 🔊 Voice Output (सुनकर जवाब) | ✅ |
| 💻 Advanced Coding Expert | ✅ |
| 🌐 Global Datasets से Training | ✅ |
| 🗄️ RAG + Training साथ | ✅ |
| 💾 Anti-Disconnect (50 steps save) | ✅ |

⚠️ **Runtime > Change Runtime Type > T4 GPU**

---
## ⚡ STEP 1: Anti-Disconnect + GPU

In [ ]:
from IPython.display import Javascript, display
import subprocess

display(Javascript('''
setInterval(function(){
    var b=document.querySelectorAll("colab-toolbar-button");
    if(b.length>0) console.log("Shiv-AI alive: "+new Date().toLocaleTimeString());
}, 55000);
console.log("Anti-disconnect ON!");
'''))

r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    for line in r.stdout.split('\n'):
        if any(x in line for x in ['MiB','T4','Tesla','GPU']):
            print(line)
    print('\n✅ GPU मिला! System start हो रहा है...')
else:
    print('❌ GPU नहीं! Runtime > T4 GPU करें!')

---
## 📦 STEP 2: सभी Libraries Install

In [ ]:
print('📦 Ultimate System libraries install हो रही हैं...')
print('(10-12 मिनट — सिर्फ पहली बार)')
print()

# Core Training
!pip install -q transformers==4.40.0 datasets==2.18.0
!pip install -q peft==0.10.0 trl==0.8.6
!pip install -q accelerate==0.29.3 bitsandbytes==0.43.1

# RAG + Search
!pip install -q faiss-cpu sentence-transformers
!pip install -q wikipedia-api duckduckgo-search
!pip install -q newspaper3k feedparser
!pip install -q beautifulsoup4 requests

# Vision
!pip install -q Pillow opencv-python-headless
!pip install -q pytesseract
!apt-get install -q tesseract-ocr tesseract-ocr-hin  # Hindi OCR

# Voice
!pip install -q SpeechRecognition
!pip install -q gTTS pydub
!apt-get install -q ffmpeg

# Data + API
!pip install -q pandas numpy scikit-learn
!pip install -q pypdf2 pdfplumber python-docx openpyxl
!pip install -q groq langdetect sentencepiece ipywidgets

print()
print('✅ सभी libraries install हुईं!')
print('  🔍 Search: Wikipedia + DuckDuckGo + News')
print('  👁️ Vision: PIL + OpenCV + Tesseract OCR')
print('  🎤 Voice : SpeechRecognition + gTTS')
print('  💻 Coding: Code execution sandbox')
print('  🧠 Brain : Transformers + LoRA + RAG')

---
## 📁 STEP 3: Drive + Setup

In [ ]:
from google.colab import drive
import os, json, pickle
from datetime import datetime

drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/ShivAI_Training'
PATHS = {
    'datasets'   : f'{BASE}/datasets',
    'uploaded'   : f'{BASE}/uploaded_datasets',
    'checkpoints': f'{BASE}/checkpoints',
    'models'     : f'{BASE}/models',
    'logs'       : f'{BASE}/logs',
    'rag_db'     : f'{BASE}/rag_database',
    'feedback'   : f'{BASE}/feedback',
    'progress'   : f'{BASE}/progress',
    'search_cache': f'{BASE}/search_cache',
    'voice_files' : f'{BASE}/voice_files',
    'vision_files': f'{BASE}/vision_files',
}
for p in PATHS.values():
    os.makedirs(p, exist_ok=True)

PROGRESS_FILE = f"{PATHS['progress']}/v5_progress.json"
print('✅ Drive ready!')
print('\n📁 ShivAI Ultimate Structure:')
for name, path in PATHS.items():
    n = len(os.listdir(path))
    print(f'  ├── {name:15} ({n} files)')

---
## 🔍 STEP 4: ONLINE SEARCH SYSTEM
### Wikipedia + DuckDuckGo + News — Real-time Data!
```
आप topic बताएं
      ↓
Wikipedia + Web + News से data आए
      ↓
Drive में save हो
      ↓
Training + RAG दोनों में use हो!
```

In [ ]:
import requests
import json
import re
import os
from bs4 import BeautifulSoup
from duckduckgo_search import DDGS
import ipywidgets as widgets
from IPython.display import display, clear_output

search_collected = []  # search से मिला data

def search_wikipedia(topic, lang='en'):
    '''Wikipedia से data लाएं — Hindi और English दोनों!'''
    results = []
    try:
        # Wikipedia API
        for language in [lang, 'hi'] if lang == 'en' else [lang]:
            url = f'https://{language}.wikipedia.org/api/rest_v1/page/summary/{topic.replace(" ","_")}'
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                data = resp.json()
                if 'extract' in data:
                    results.append({
                        'title'  : data.get('title',''),
                        'text'   : data['extract'],
                        'source' : f'wikipedia_{language}',
                        'url'    : data.get('content_urls',{}).get('desktop',{}).get('page',''),
                        'type'   : 'search'
                    })
        # Full article भी लाएं
        full_url = f'https://en.wikipedia.org/w/api.php?action=query&titles={topic}&prop=extracts&explaintext=1&format=json'
        resp2 = requests.get(full_url, timeout=10)
        if resp2.status_code == 200:
            pages = resp2.json().get('query',{}).get('pages',{})
            for page in pages.values():
                text = page.get('extract','')
                if text and len(text) > 200:
                    results.append({
                        'title' : page.get('title',''),
                        'text'  : text[:5000],
                        'source': 'wikipedia_full',
                        'type'  : 'search'
                    })
    except Exception as e:
        print(f'  Wikipedia error: {e}')
    return results

def search_web(query, max_results=5):
    '''DuckDuckGo से web search'''
    results = []
    try:
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=max_results):
                results.append({
                    'title' : r.get('title',''),
                    'text'  : r.get('body',''),
                    'url'   : r.get('href',''),
                    'source': 'web_search',
                    'type'  : 'search'
                })
    except Exception as e:
        print(f'  Web search error: {e}')
    return results

def search_news(topic, max_results=5):
    '''Latest news ढूंढें'''
    results = []
    try:
        with DDGS() as ddgs:
            for r in ddgs.news(topic, max_results=max_results):
                results.append({
                    'title' : r.get('title',''),
                    'text'  : r.get('body',''),
                    'url'   : r.get('url',''),
                    'date'  : r.get('date',''),
                    'source': 'news',
                    'type'  : 'search'
                })
    except Exception as e:
        print(f'  News error: {e}')
    return results

def save_search_to_drive(results, topic):
    '''Search results Drive में save करें'''
    fname = re.sub(r'[^a-zA-Z0-9_]', '_', topic) + '_search.json'
    fpath = os.path.join(PATHS['search_cache'], fname)
    existing = []
    if os.path.exists(fpath):
        with open(fpath) as f: existing = json.load(f)
    existing.extend(results)
    with open(fpath, 'w', encoding='utf-8') as f:
        json.dump(existing, f, ensure_ascii=False, indent=2)
    return fpath

# ── Search UI ─────────────────────────────────────
topic_input = widgets.Text(
    placeholder='Topic लिखें (जैसे: Python programming, Ayurveda, Cybersecurity)',
    layout=widgets.Layout(width='500px')
)
search_types = widgets.SelectMultiple(
    options=['Wikipedia', 'Web Search', 'News'],
    value=['Wikipedia', 'Web Search'],
    description='Sources:',
    layout=widgets.Layout(height='80px')
)
add_to_train = widgets.Checkbox(value=True, description='Training data में add करें')
add_to_rag   = widgets.Checkbox(value=True, description='RAG database में add करें')
search_btn   = widgets.Button(description='🔍 Search करें',
                               button_style='primary',
                               layout=widgets.Layout(width='160px', height='42px'))
search_out   = widgets.Output()

def on_search(b):
    topic = topic_input.value.strip()
    if not topic:
        with search_out: print('❌ Topic लिखें!')
        return
    with search_out:
        clear_output()
        print(f'🔍 Searching: "{topic}"')
        print('='*50)
        all_results = []

        if 'Wikipedia' in search_types.value:
            print('📖 Wikipedia...')
            wiki_results = search_wikipedia(topic)
            all_results.extend(wiki_results)
            print(f'   ✅ {len(wiki_results)} articles मिले')

        if 'Web Search' in search_types.value:
            print('🌐 Web Search...')
            web_results = search_web(topic)
            all_results.extend(web_results)
            print(f'   ✅ {len(web_results)} results मिले')

        if 'News' in search_types.value:
            print('📰 News...')
            news_results = search_news(topic)
            all_results.extend(news_results)
            print(f'   ✅ {len(news_results)} news मिले')

        if all_results:
            fpath = save_search_to_drive(all_results, topic)
            search_collected.extend(all_results)
            print(f'\n✅ Total: {len(all_results)} results')
            print(f'💾 Drive में save: search_cache/{os.path.basename(fpath)}')
            print(f'\n📄 Preview:')
            for r in all_results[:2]:
                print(f'  [{r["source"]}] {r["title"]}')
                print(f'  {r["text"][:200]}...')
                print()
        else:
            print('❌ कोई result नहीं मिला')

search_btn.on_click(on_search)

print('🔍 ONLINE SEARCH SYSTEM')
print('Wikipedia + Web + News से real-time data!')
display(topic_input)
display(search_types)
display(widgets.HBox([add_to_train, add_to_rag]))
display(search_btn)
display(search_out)

---
## 👁️ STEP 5: VISION SYSTEM
### Image/Screenshot देखकर समझना
```
Screenshot upload करें
        ↓
OCR से text निकाला
        ↓
Vision Model analyze करे
        ↓
"इस code में line 5 पर error है!"
```

In [ ]:
from PIL import Image
import pytesseract
import cv2
import numpy as np
import io
import base64
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import files as colab_files

vision_context = {'last_image_text': '', 'last_image_path': ''}

def preprocess_image(img):
    '''Image को better OCR के लिए तैयार करें'''
    img_array = np.array(img)
    # Grayscale
    if len(img_array.shape) == 3:
        gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    else:
        gray = img_array
    # Contrast बढ़ाएं
    enhanced = cv2.equalizeHist(gray)
    # Denoise
    denoised = cv2.fastNlMeansDenoising(enhanced, h=10)
    return Image.fromarray(denoised)

def extract_text_from_image(img, langs='eng+hin'):
    '''Image से text निकालें — Hindi + English दोनों'''
    try:
        # Normal OCR
        text_normal = pytesseract.image_to_string(img, lang=langs)
        # Enhanced OCR
        enhanced = preprocess_image(img)
        text_enhanced = pytesseract.image_to_string(enhanced, lang=langs)
        # जो longer है वो better है
        text = text_normal if len(text_normal) > len(text_enhanced) else text_enhanced
        return text.strip()
    except Exception as e:
        return f'OCR Error: {e}'

def analyze_image_type(text):
    '''Image किस type की है — detect करें'''
    text_lower = text.lower()
    if any(w in text_lower for w in ['def ', 'function', 'class ', 'import ', 'error:', 'traceback']):
        return 'code', '💻 Code/Error Screenshot'
    elif any(w in text_lower for w in ['http', 'www', 'html', 'css', '<div', '<p>']):
        return 'web', '🌐 Website Screenshot'
    elif any(w in text_lower for w in ['select', 'from', 'where', 'table', 'database']):
        return 'database', '🗄️ Database/SQL'
    elif len(text) < 50:
        return 'image', '🖼️ Image (कम text)'
    else:
        return 'document', '📄 Document/Text'

# ── Vision UI ─────────────────────────────────────
upload_img_btn = widgets.Button(
    description='📸 Image Upload करें',
    button_style='info',
    layout=widgets.Layout(width='200px', height='42px')
)
vision_question = widgets.Text(
    placeholder='इस image के बारे में क्या पूछना है? (खाली छोड़ें = auto analyze)',
    layout=widgets.Layout(width='100%')
)
analyze_btn = widgets.Button(
    description='👁️ Analyze करें',
    button_style='success',
    layout=widgets.Layout(width='160px', height='42px')
)
vision_out = widgets.Output()

def on_upload_image(b):
    with vision_out:
        clear_output()
        print('📸 Image select करें (PNG/JPG/Screenshot)...')
        uploaded = colab_files.upload()
        for fname, content in uploaded.items():
            try:
                img = Image.open(io.BytesIO(content))
                print(f'\n✅ Image loaded: {fname}')
                print(f'   Size: {img.size[0]}×{img.size[1]} pixels')

                # Save to drive
                save_path = os.path.join(PATHS['vision_files'], fname)
                img.save(save_path)
                vision_context['last_image_path'] = save_path

                # OCR
                print('\n🔍 Text extract हो रहा है...')
                text = extract_text_from_image(img)
                img_type, type_label = analyze_image_type(text)
                vision_context['last_image_text'] = text
                vision_context['image_type'] = img_type

                print(f'   Type: {type_label}')
                if text:
                    print(f'   Text found ({len(text)} chars):')
                    print(f'   "{text[:300]}"')
                else:
                    print('   (Text नहीं मिला — pure image है)')
                print('\n✅ Analyze करने के लिए नीचे button click करें!')
            except Exception as e:
                print(f'❌ Error: {e}')

def on_analyze(b):
    with vision_out:
        if not vision_context['last_image_text'] and not vision_context['last_image_path']:
            print('❌ पहले image upload करें!')
            return

        img_text = vision_context.get('last_image_text', '')
        img_type = vision_context.get('image_type', 'document')
        question = vision_question.value.strip()

        # Vision analysis prompt बनाएं
        if img_type == 'code':
            auto_prompt = f'इस code को analyze करो, errors ढूंढो, और better solution दो:\n{img_text}'
        elif img_type == 'web':
            auto_prompt = f'इस website screenshot को analyze करो:\n{img_text}'
        else:
            auto_prompt = f'इस image का content explain करो:\n{img_text}'

        final_prompt = question if question else auto_prompt
        print(f'\n👁️ Vision Analysis:')
        print(f'Type: {img_type}')
        print('─'*50)
        # यह result Step 10 के chat में जाएगा
        vision_context['pending_question'] = final_prompt
        print(f'✅ Question ready: "{final_prompt[:100]}"')
        print('\nStep 10 में Chat खोलें — image analysis का जवाब मिलेगा!')

upload_img_btn.on_click(on_upload_image)
analyze_btn.on_click(on_analyze)

print('👁️ VISION SYSTEM')
print('Screenshot/Image upload करें — Shiv-AI देखकर समझेगा!')
display(upload_img_btn)
display(vision_question)
display(analyze_btn)
display(vision_out)

---
## 🎤 STEP 6: VOICE SYSTEM
### बोलकर पूछो + सुनकर जवाब पाओ!
```
🎤 Microphone → Speech-to-Text → Shiv-AI → Text-to-Speech → 🔊
```

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display as ipy_display
import ipywidgets as widgets
from IPython.display import display, clear_output, Javascript
import os, base64, tempfile

voice_context = {'last_transcript': '', 'voice_enabled': True}

# ── Text to Speech ────────────────────────────────
def text_to_speech(text, lang='hi', save_path=None):
    '''
    Text → Voice
    lang: hi = Hindi, en = English
    '''
    try:
        # Language detect करें
        hindi_chars = sum(1 for c in text if '\u0900' <= c <= '\u097f')
        detected_lang = 'hi' if hindi_chars > len(text) * 0.2 else 'en'
        use_lang = lang if lang != 'auto' else detected_lang

        tts = gTTS(text=text[:500], lang=use_lang, slow=False)
        if save_path:
            tts.save(save_path)
            return save_path
        else:
            tmp = tempfile.NamedTemporaryFile(suffix='.mp3', delete=False)
            tts.save(tmp.name)
            return tmp.name
    except Exception as e:
        print(f'TTS Error: {e}')
        return None

def play_voice(text, lang='auto'):
    '''Text को voice में play करें'''
    if not voice_context['voice_enabled']:
        return
    audio_path = text_to_speech(text, lang)
    if audio_path:
        ipy_display(Audio(audio_path, autoplay=True))

# ── Speech to Text (Browser Microphone) ──────────
# Colab में microphone के लिए JavaScript use करना होगा
voice_record_js = '''
async function recordVoice() {
    const stream = await navigator.mediaDevices.getUserMedia({audio: true});
    const recorder = new MediaRecorder(stream);
    const chunks = [];

    recorder.ondataavailable = e => chunks.push(e.data);
    recorder.onstop = async () => {
        const blob = new Blob(chunks, {type: 'audio/webm'});
        const reader = new FileReader();
        reader.onload = () => {
            const base64 = reader.result.split(',')[1];
            google.colab.kernel.invokeFunction('notebook.processAudio', [base64], {});
        };
        reader.readAsDataURL(blob);
    };

    recorder.start();
    document.getElementById('voice-status').textContent = '🔴 Recording...';

    setTimeout(() => {
        recorder.stop();
        stream.getTracks().forEach(t => t.stop());
        document.getElementById('voice-status').textContent = '⏳ Processing...';
    }, 5000);  // 5 seconds record
}
'''

# ── Voice UI ──────────────────────────────────────
voice_toggle = widgets.ToggleButton(
    value=True,
    description='🔊 Voice ON',
    button_style='success',
    layout=widgets.Layout(width='130px', height='40px')
)
lang_select = widgets.Dropdown(
    options=[('Auto Detect', 'auto'), ('Hindi', 'hi'), ('English', 'en')],
    value='auto', description='Language:',
    layout=widgets.Layout(width='200px')
)
test_voice_btn = widgets.Button(
    description='🎵 Voice Test करें',
    button_style='info',
    layout=widgets.Layout(width='170px', height='40px')
)
voice_out = widgets.Output()

def on_voice_toggle(change):
    voice_context['voice_enabled'] = change['new']
    voice_toggle.description = '🔊 Voice ON' if change['new'] else '🔇 Voice OFF'
    voice_toggle.button_style = 'success' if change['new'] else 'danger'

def on_test_voice(b):
    with voice_out:
        clear_output()
        print('🎵 Voice test हो रहा है...')
        test_texts = [
            ('hi', 'नमस्ते! मैं Shiv-AI हूं। आप मुझसे कोई भी सवाल पूछ सकते हैं।'),
            ('en', 'Hello! I am Shiv-AI, your intelligent assistant.'),
        ]
        for lang, text in test_texts:
            print(f'[{lang}]: {text}')
            play_voice(text, lang)
        print('✅ Voice system ready!')

voice_toggle.observe(on_voice_toggle, names='value')
test_voice_btn.on_click(on_test_voice)

print('🎤 VOICE SYSTEM')
print('Text-to-Speech: Hindi + English दोनों!')
print('(Microphone input के लिए Step 10 का chat use करें)')
display(widgets.HBox([voice_toggle, lang_select, test_voice_btn]))
display(voice_out)

---
## 💻 STEP 7: ADVANCED CODING EXPERT
### Code लिखे + Test करे + खुद Fix करे!
```
आप: "Bubble sort लिखो"
        ↓
Code लिखा + Run किया
        ↓
Error आई? खुद fix किया!
        ↓
Perfect code दिया!
```

In [ ]:
import subprocess
import tempfile
import os
import sys
import ipywidgets as widgets
from IPython.display import display, clear_output
import io
from contextlib import redirect_stdout, redirect_stderr

code_context = {'last_code': '', 'last_output': '', 'iteration': 0}

def safe_execute_code(code, timeout=10):
    '''
    Code को safely execute करें
    - Dangerous commands block होंगे
    - Timeout है
    - Output capture होगा
    '''
    # Dangerous commands check
    dangerous = ['os.system', 'subprocess', 'shutil.rmtree',
                 '__import__', 'eval(', 'exec(', 'open("/etc']
    for d in dangerous:
        if d in code:
            return f'⚠️ Blocked: "{d}" dangerous command है', False

    stdout_capture = io.StringIO()
    stderr_capture = io.StringIO()

    try:
        with redirect_stdout(stdout_capture), redirect_stderr(stderr_capture):
            exec(code, {'__name__': '__main__'})
        output = stdout_capture.getvalue()
        error  = stderr_capture.getvalue()
        if error:
            return f'Output:\n{output}\nWarnings:\n{error}', True
        return output or '✅ Code ran successfully (no output)', True
    except Exception as e:
        return f'❌ Error: {type(e).__name__}: {str(e)}', False

def format_code_response(raw_response):
    '''LLM response से code extract करें'''
    # Code blocks ढूंढें
    import re
    patterns = [
        r'```python\n(.*?)```',
        r'```\n(.*?)```',
        r'`(.*?)`',
    ]
    for pattern in patterns:
        matches = re.findall(pattern, raw_response, re.DOTALL)
        if matches:
            return matches[0].strip()
    return raw_response.strip()

# ── Coding UI ─────────────────────────────────────
code_input = widgets.Textarea(
    placeholder='Code request लिखें... (जैसे: bubble sort in Python, या अपना code paste करें)',
    layout=widgets.Layout(width='100%', height='100px')
)
lang_drop = widgets.Dropdown(
    options=['Python', 'JavaScript', 'Java', 'C++', 'SQL', 'HTML/CSS', 'Bash'],
    value='Python', description='Language:',
    layout=widgets.Layout(width='200px')
)
mode_drop = widgets.Dropdown(
    options=['Write Code', 'Fix Error', 'Explain Code', 'Optimize', 'Add Comments'],
    value='Write Code', description='Mode:',
    layout=widgets.Layout(width='220px')
)
run_toggle = widgets.Checkbox(value=True, description='Code run करके test करें')
code_btn   = widgets.Button(
    description='💻 Code Generate करें',
    button_style='success',
    layout=widgets.Layout(width='200px', height='42px')
)
code_out = widgets.Output()

# यह function Step 10 में Groq के साथ connect होगा
coding_requests = []  # pending coding requests

def on_code_request(b):
    req = code_input.value.strip()
    if not req:
        with code_out: print('❌ Request लिखें!')
        return
    with code_out:
        clear_output()
        lang = lang_drop.value
        mode = mode_drop.value
        coding_requests.append({'request': req, 'lang': lang, 'mode': mode, 'run': run_toggle.value})
        print(f'💻 Coding Request:')
        print(f'   Language: {lang}')
        print(f'   Mode    : {mode}')
        print(f'   Request : {req}')
        print(f'   Auto-run: {run_toggle.value}')
        print('\n✅ Request queued! Step 10 में Chat खोलें।')

code_btn.on_click(on_code_request)

print('💻 ADVANCED CODING EXPERT')
print('Code लिखे + Run करे + Error Fix करे!')
display(code_input)
display(widgets.HBox([lang_drop, mode_drop]))
display(widgets.HBox([run_toggle, code_btn]))
display(code_out)

---
## 🗃️ STEP 8: DATA PIPELINE
### Search + Upload + Chunking + Tokenization — सब एक जगह

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset
import pandas as pd
import random, re
import pdfplumber
from google.colab import files as colab_files
import ipywidgets as widgets
from IPython.display import display, clear_output

BASE_MODEL = 'microsoft/phi-2'
MAX_LEN    = 512

print(f'📥 Tokenizer load: {BASE_MODEL}')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'
print('✅ Tokenizer ready!')

# Smart Chunker
def smart_chunk(text, size=500, overlap=100):
    text = re.sub(r'\s+', ' ', text.strip())
    if len(text) <= size: return [text] if len(text) > 50 else []
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            for p in ['।','।\n','.\n','\n\n','. ']:
                pos = text.rfind(p, start + size//2, end)
                if pos > start + size//2: end = pos + len(p); break
        c = text[start:end].strip()
        if len(c) > 50: chunks.append(c)
        start = end - overlap
    return chunks

def to_training_format(doc, bot='general'):
    dtype = doc.get('type', 'text')
    if dtype == 'qa':
        i = doc.get('instruction', doc.get('question',''))
        o = doc.get('output', doc.get('answer',''))
        inp = doc.get('input','')
        t = f'### Bot: {bot}\n### Instruction:\n{i}'
        if inp and inp != i: t += f'\n### Input:\n{inp}'
        return [t + f'\n\n### Response:\n{o}']
    elif dtype in ['pdf','text','web','search']:
        return [f'### Bot: {bot}\n### Context:\n{c}\n\n### Response:\nI understand.' for c in smart_chunk(doc.get('text',''))]
    return []

# ── Upload UI ──────────────────────────────────────
upload_raw = []
ul_btn  = widgets.Button(description='📁 Upload File', button_style='success',
                          layout=widgets.Layout(width='160px',height='40px'))
can_btn = widgets.Button(description='❌ Cancel', button_style='danger',
                          layout=widgets.Layout(width='100px',height='40px'))
clr_btn = widgets.Button(description='🗑️ Clear', button_style='warning',
                          layout=widgets.Layout(width='90px',height='40px'))
ul_out  = widgets.Output()

def on_ul(b):
    with ul_out:
        clear_output()
        print('📂 Select: JSON / CSV / PDF / TXT / Excel')
        uploaded = colab_files.upload()
        for fname, content in uploaded.items():
            ext = fname.lower().rsplit('.',1)[-1]
            docs = []
            try:
                if ext == 'pdf':
                    with pdfplumber.open(io.BytesIO(content)) as pdf:
                        for i,pg in enumerate(pdf.pages):
                            t = pg.extract_text()
                            if t: docs.append({'text':t,'source':fname,'type':'pdf'})
                elif ext in ['json','jsonl']:
                    data = json.loads(content.decode('utf-8'))
                    if not isinstance(data,list): data=[data]
                    for item in data:
                        i=item.get('instruction',item.get('question',item.get('input','')))
                        o=item.get('output',item.get('answer',''))
                        if i and o: docs.append({'instruction':i,'output':o,'source':fname,'type':'qa'})
                elif ext == 'csv':
                    df = pd.read_csv(io.StringIO(content.decode('utf-8'))).dropna(how='all')
                    for _,row in df.iterrows():
                        i=str(row.iloc[0]); o=str(row.iloc[1]) if len(row)>1 else ''
                        if len(i)>10 and len(o)>10: docs.append({'instruction':i,'output':o,'source':fname,'type':'qa'})
                elif ext in ['xlsx','xls']:
                    df = pd.read_excel(io.BytesIO(content)).dropna(how='all')
                    for _,row in df.iterrows():
                        i=str(row.iloc[0]); o=str(row.iloc[1]) if len(row)>1 else ''
                        if len(i)>10 and len(o)>10: docs.append({'instruction':i,'output':o,'source':fname,'type':'qa'})
                elif ext == 'txt':
                    docs.append({'text':content.decode('utf-8'),'source':fname,'type':'text'})
            except Exception as e:
                print(f'  ❌ {fname}: {e}'); continue
            upload_raw.extend(docs)
            sp = os.path.join(PATHS['uploaded'], fname.rsplit('.',1)[0]+'_parsed.json')
            with open(sp,'w',encoding='utf-8') as f: json.dump(docs,f,ensure_ascii=False,indent=2)
            print(f'  ✅ {fname}: {len(docs)} examples → Drive saved')
        print(f'\nTotal: {len(upload_raw)} examples')

ul_btn.on_click(on_ul)
can_btn.on_click(lambda b: ul_out.clear_output())
clr_btn.on_click(lambda b: (upload_raw.clear(), ul_out.clear_output()))

print('📤 UPLOAD + TOKENIZATION PIPELINE')
display(widgets.HBox([ul_btn, can_btn, clr_btn]))
display(ul_out)

print('\n🔢 Tokenization Demo:')
for label, text in [('Hindi','Python में function कैसे लिखते हैं?'),
                     ('English','Write a function in Python'),
                     ('Code','def hello(): print("Hi")')]:
    toks = tokenizer.encode(text)
    print(f'  [{label}] "{text}" → {toks[:8]}... ({len(toks)} tokens)')

---
## 🏋️ STEP 9: TRAINING + RAG एक साथ

In [ ]:
import torch, faiss, pickle, random
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, DataCollatorForLanguageModeling, TrainerCallback
from peft import PeftModel, LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer
from sentence_transformers import SentenceTransformer
from datasets import Dataset

# ── Master Dataset बनाएं ──────────────────────────
all_texts = []

# 1. Uploaded docs
for doc in upload_raw:
    all_texts.extend([{'text':t,'source':doc.get('source','upload')} for t in to_training_format(doc)])

# 2. Search results
for doc in search_collected:
    all_texts.extend([{'text':t,'source':'search'} for t in to_training_format(doc)])

# 3. Drive datasets
for folder in [PATHS['datasets'], PATHS['uploaded'], PATHS['search_cache']]:
    for fname in os.listdir(folder):
        if not fname.endswith('.json'): continue
        try:
            with open(os.path.join(folder,fname),'r',encoding='utf-8') as f: data=json.load(f)
            if not isinstance(data,list): continue
            bot = fname.replace('.json','').replace('_parsed','').replace('hf_','')[:20]
            for item in data:
                for t in to_training_format(item, bot): all_texts.append({'text':t,'source':fname})
        except: pass

# Filter + shuffle
filtered = [x for x in all_texts if 20 <= len(tokenizer.encode(x['text'], truncation=True, max_length=MAX_LEN)) <= MAX_LEN]
random.shuffle(filtered)
final_dataset = Dataset.from_list(filtered)

print(f'✅ Dataset: {len(final_dataset)} examples')

# ── Model Load ────────────────────────────────────
print(f'\n📥 Model loading: {BASE_MODEL}')
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                          bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                              device_map='auto', torch_dtype=torch.float16, trust_remote_code=True)
model = prepare_model_for_kbit_training(model)

adapter_ok = any(os.path.exists(os.path.join(PATHS['models'],f)) for f in ['adapter_model.safetensors','adapter_model.bin'])
if adapter_ok:
    print('🔄 Resume!')
    model = PeftModel.from_pretrained(model, PATHS['models'], is_trainable=True)
else:
    print('🆕 New LoRA')
    model = get_peft_model(model, LoraConfig(task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32,
        target_modules=['q_proj','v_proj','k_proj','o_proj'], lora_dropout=0.05, bias='none'))

# ── Training ──────────────────────────────────────
class SaveCallback(TrainerCallback):
    def __init__(self, pf):
        self.pf = pf; self.t0 = datetime.now()
    def on_save(self, args, state, control, **kwargs):
        m = (datetime.now()-self.t0).seconds//60
        with open(self.pf,'w') as f: json.dump({'step':state.global_step,'mins':m,'date':datetime.now().strftime('%Y-%m-%d %H:%M')},f)
        print(f'  💾 Step {state.global_step} saved | {m}min')
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            pct = state.global_step/state.max_steps*100 if state.max_steps else 0
            print(f'  📈 {state.global_step}/{state.max_steps} ({pct:.0f}%) Loss:{logs["loss"]:.4f}')

args = TrainingArguments(
    output_dir=PATHS['checkpoints'], num_train_epochs=3,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    learning_rate=2e-4, fp16=True, logging_steps=25,
    save_steps=50, save_total_limit=5,  # Anti-disconnect!
    warmup_ratio=0.03, lr_scheduler_type='cosine',
    report_to='none', overwrite_output_dir=False, dataloader_pin_memory=False
)
trainer = SFTTrainer(model=model, args=args, train_dataset=final_dataset,
    tokenizer=tokenizer, dataset_text_field='text', max_seq_length=MAX_LEN,
    data_collator=DataCollatorForLanguageModeling(tokenizer,mlm=False),
    callbacks=[SaveCallback(PROGRESS_FILE)])

cps = sorted([d for d in os.listdir(PATHS['checkpoints']) if d.startswith('checkpoint-')], key=lambda x:int(x.split('-')[1]))
resume = os.path.join(PATHS['checkpoints'],cps[-1]) if cps else None
print(f'\n🚀 Training... ({"Resume: "+cps[-1] if resume else "New"}) | Save every 50 steps!')
trainer.train(resume_from_checkpoint=resume)
trainer.save_model(PATHS['models'])
tokenizer.save_pretrained(PATHS['models'])

# ── RAG Build ─────────────────────────────────────
print('\n🗄️ RAG Database building...')
embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
rag_chunks = []
for doc in upload_raw + search_collected:
    dtype = doc.get('type','text')
    if dtype in ['pdf','text','web','search']:
        for i,c in enumerate(smart_chunk(doc.get('text',''))):
            rag_chunks.append({'text':c,'source':doc.get('source','?'),'chunk_id':i})
    elif dtype == 'qa':
        t = f"{doc.get('instruction','')} {doc.get('output','')}"
        if len(t)>50: rag_chunks.append({'text':t[:500],'source':doc.get('source','?'),'chunk_id':0})

DB_F  = os.path.join(PATHS['rag_db'],'v5.index')
CHK_F = os.path.join(PATHS['rag_db'],'v5_chunks.pkl')
if rag_chunks:
    embs = embedder.encode([c['text'] for c in rag_chunks], batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    if os.path.exists(DB_F):
        idx = faiss.read_index(DB_F)
        with open(CHK_F,'rb') as f: saved_chunks = pickle.load(f)
        idx.add(embs.astype('float32')); saved_chunks.extend(rag_chunks)
    else:
        idx = faiss.IndexFlatL2(embs.shape[1]); idx.add(embs.astype('float32')); saved_chunks = rag_chunks
    faiss.write_index(idx, DB_F)
    with open(CHK_F,'wb') as f: pickle.dump(saved_chunks,f)
    print(f'✅ RAG: {idx.ntotal} vectors | {len(saved_chunks)} chunks')
else:
    print('⚠️ RAG: कोई document नहीं — upload करें')
    idx = None; saved_chunks = []

---
## 💬 STEP 10: MASTER CHAT
### Voice + Vision + Search + Coding + RAG — सब एक Chat में!

In [ ]:
from groq import Groq
import ipywidgets as widgets
from IPython.display import display, clear_output, Audio

GROQ_API_KEY = 'your_groq_api_key_here'  # ← console.groq.com (Free!)

BOT_SYSTEM = {
    'coding'  : 'तुम expert programmer हो। Code लिखो, explain करो, errors fix करो। हमेशा working code दो।',
    'cyber'   : 'तुम ethical cybersecurity expert हो।',
    'ayurveda': 'तुम आयुर्वेद विशेषज्ञ हो। Hindi में जवाब दो।',
    'myth'    : 'तुम Hindu mythology expert हो।',
    'content' : 'तुम creative content writer हो।',
    'general' : 'तुम Shiv-AI हो — helpful, accurate assistant।',
}

def detect_bot(q):
    q = q.lower()
    if any(w in q for w in ['code','python','java','function','bug','error','program','write','sql','html']): return 'coding'
    if any(w in q for w in ['hack','cyber','security','attack','firewall','osint']): return 'cyber'
    if any(w in q for w in ['ayurveda','आयुर्वेद','herb','तुलसी','health']): return 'ayurveda'
    if any(w in q for w in ['myth','ramayan','mahabharat','god','देव','कथा']): return 'myth'
    if any(w in q for w in ['write','blog','article','content']): return 'content'
    return 'general'

def rag_get(query, top_k=3):
    if idx is None or not saved_chunks: return ''
    qe = embedder.encode([query], convert_to_numpy=True)
    ds, ids = idx.search(qe.astype('float32'), top_k)
    return '\n\n'.join([saved_chunks[i]['text'] for d,i in zip(ds[0],ids[0]) if i<len(saved_chunks) and 1/(1+d)>0.25])

if GROQ_API_KEY != 'your_groq_api_key_here':
    client  = Groq(api_key=GROQ_API_KEY)
    history = []
    last_qa = {'q':'','a':''}

    def master_ask(question, bot=None, image_context='', web_search=False):
        bot = bot or detect_bot(question)
        sys = BOT_SYSTEM.get(bot, BOT_SYSTEM['general'])

        # RAG
        rag_ctx = rag_get(question)
        if rag_ctx: sys += f'\n\n📚 Knowledge Base:\n{rag_ctx}'

        # Vision context
        if image_context: sys += f'\n\n👁️ Image Content:\n{image_context}'

        # Web search context
        if web_search:
            web = search_web(question, max_results=3)
            if web: sys += '\n\n🔍 Web Results:\n' + '\n'.join([r['text'][:200] for r in web])

        # Code mode
        if bot == 'coding':
            sys += '\n\nAlways provide complete, runnable code. Include comments.'

        msgs = [{'role':'system','content':sys}, *history[-8:], {'role':'user','content':question}]
        resp = client.chat.completions.create(model='llama-3.1-70b-versatile',
                                               messages=msgs, max_tokens=1500, temperature=0.2)
        ans = resp.choices[0].message.content
        history.extend([{'role':'user','content':question},{'role':'assistant','content':ans}])

        # Code auto-run
        if bot == 'coding' and coding_requests and coding_requests[-1].get('run', False):
            code = format_code_response(ans)
            if code and 'def ' in code or 'print' in code:
                result, success = safe_execute_code(code)
                ans += f'\n\n▶️ Execution Result:\n```\n{result}\n```'

        return ans, bot, bool(rag_ctx)

    # ── Master UI ──────────────────────────────────
    q_box    = widgets.Textarea(placeholder='सवाल लिखें (Hindi/English/Code)...',
                                 layout=widgets.Layout(width='100%',height='80px'))
    bot_drop = widgets.Dropdown(options=['auto']+list(BOT_SYSTEM.keys()),
                                 value='auto', description='Bot:')
    web_chk  = widgets.Checkbox(value=False, description='🔍 Web Search भी करें')
    img_chk  = widgets.Checkbox(value=True, description='👁️ Image context use करें')
    voice_chk= widgets.Checkbox(value=True, description='🔊 Voice में जवाब')
    ask_btn  = widgets.Button(description='🤖 पूछें', button_style='primary',
                               layout=widgets.Layout(width='110px',height='42px'))
    good_btn = widgets.Button(description='👍 सही', button_style='success',
                               layout=widgets.Layout(width='95px',height='35px'))
    bad_btn  = widgets.Button(description='👎 गलत', button_style='danger',
                               layout=widgets.Layout(width='95px',height='35px'))
    clr_btn  = widgets.Button(description='🗑️ Clear', button_style='warning',
                               layout=widgets.Layout(width='85px',height='35px'))
    chat_out = widgets.Output()

    def on_ask(b):
        q = q_box.value.strip()
        # Pending coding request?
        if not q and coding_requests:
            cr = coding_requests[-1]
            q = f"Write {cr['lang']} code: {cr['request']}"
        # Pending vision?
        if not q and vision_context.get('pending_question'):
            q = vision_context['pending_question']
        if not q: return
        q_box.value = ''
        bot = None if bot_drop.value=='auto' else bot_drop.value
        img_ctx = vision_context.get('last_image_text','') if img_chk.value else ''

        with chat_out:
            print(f'\n❓ {q}')
            ans, used_bot, used_rag = master_ask(q, bot, img_ctx, web_chk.value)
            rag_lbl = '📚 RAG' if used_rag else '🧠'
            web_lbl = '🔍 Web' if web_chk.value else ''
            img_lbl = '👁️ Vision' if img_ctx else ''
            labels  = ' | '.join(filter(None, [used_bot, rag_lbl, web_lbl, img_lbl]))
            print(f'🤖 [{labels}]')
            print(ans)
            print('─'*55)
            last_qa.update({'q':q,'a':ans})
            # Voice output
            if voice_chk.value and voice_context.get('voice_enabled', True):
                clean_ans = re.sub(r'```.*?```','(code)',ans,flags=re.DOTALL)[:400]
                play_voice(clean_ans)

    def save_fb(rating):
        if not last_qa['q']: return
        fb = {**last_qa,'rating':rating,'date':datetime.now().strftime('%Y-%m-%d %H:%M')}
        fp = os.path.join(PATHS['feedback'],'feedback.json')
        ex = json.load(open(fp)) if os.path.exists(fp) else []
        ex.append(fb)
        with open(fp,'w',encoding='utf-8') as f: json.dump(ex,f,ensure_ascii=False,indent=2)
        with chat_out: print(f'✅ Feedback saved ({rating}/5)')

    ask_btn.on_click(on_ask)
    good_btn.on_click(lambda b: save_fb(5))
    bad_btn.on_click(lambda b: save_fb(1))
    clr_btn.on_click(lambda b: (history.clear(), chat_out.clear_output()))

    print('💬 SHIV-AI ULTIMATE CHAT v5')
    print('Voice + Vision + Search + Coding + RAG — सब एक जगह!')
    print()
    display(q_box)
    display(widgets.HBox([ask_btn, bot_drop]))
    display(widgets.HBox([web_chk, img_chk, voice_chk]))
    display(widgets.HBox([good_btn, bad_btn, clr_btn]))
    display(chat_out)
else:
    print('⚠️ GROQ_API_KEY डालें!')
    print('   👉 https://console.groq.com → Free → API Keys → Create')